## Document RAG analysis with Closed Source LLMs

In [177]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
load_dotenv()

def create_chunks(file_path):
    # Load Document
    with open(file_path) as f:
        doc = f.read()
    
    # Split Document
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap=100,
        length_function=len
    )    

    # Create chunks
    chunks = text_splitter.split_text(doc)

    return chunks

file_path = './files/ipaybtc-faq.txt'

chunks = create_chunks(file_path)

In [178]:
# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Load embedding model
embedding_model = HuggingFaceEmbeddings(model='BAAI/bge-small-en-v1.5')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2866.11it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [179]:
# Vector store
from langchain_chroma import Chroma
import numpy as np

def create_vector_store(chunks):
    vector_store = Chroma.from_texts(
        chunks, 
        embedding=embedding_model, 
        persist_directory="./db", 
        collection_name="faq_docs"
    )
    
    return vector_store

create_vector_store(chunks)


print(f"Chroma vector store created with {len(chunks)} vectors.")

Chroma vector store created with 119 vectors.


In [180]:
# Open AI LLM
import os
from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY) 

In [181]:
from langchain_chroma import Chroma

vector_store = Chroma(
    embedding_function=embedding_model, 
    persist_directory="./db", 
    collection_name="faq_docs"
)

print(vector_store._collection.count())


def get_context(prompt):
    query_embedding = embedding_model.embed_query(prompt)
    docs = vector_store.similarity_search_with_score(prompt, k=5)
    
    # print(docs)
    
    docs = [doc for doc, score in docs if score < 0.5]
        
    context = "\n".join([doc.page_content for doc in docs])
    
    return context

482


In [182]:


conversations = []
def generate_answer_openai(prompt):
    try:
        context = get_context(prompt)
        
        print(f"Question: {prompt} \n\n")
        print(f"Context: {context} \n\n")
    
        conversations.append({"role": "user", "content": prompt})
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system", 
                    "content": """
                        You are Debbie, a friendly customer support assistant for iPayBTC.
                        Speak in a warm, conversational tone like a real human support agent.

                        Rules:
                        - This user name is Jesse
                        - If the user sends a greeting (e.g. "Hello", "Hi", "Hey"), respond warmly and ask how you can help
                        - Answer ONLY using the provided context for all other questions
                        - If the context doesn't contain the answer, say: "I'm sorry, I don't have that information. Please contact our support team for help! 😊"
                        - DO NOT generate answers from your own knowledge
                        - Use simple, friendly language
                        - Use occasional emojis where appropriate 😊
                        - Only greet if the message is JUST a greeting with no question
                        - If the user asks a question, answer directly without greeting
                    """
                 },
                {"role": "user", "content": f"Question: {prompt}\n\nContext:\n{context}"},
            ],
            temperature=0.7,
            max_tokens=150,
        )
        
        response = response.choices[0].message.content.strip()
        
        conversations.append({"role": "assistant", "content": response})
            
        return response
    except Exception as e:
        print(f"Error initializing OpenAI client: {e}")   

In [183]:
# Store past 3 questions
# Rewrite/summarize past questions to improve retrieval
# Detect follow up questions, use history of previous questions and answers.

In [192]:
prompt = "Can I trade USDT or just Bitcoin?"  # Test greeting
answer = generate_answer_openai(prompt)
print(f"Answer: {answer}")

Question: Can I trade USDT or just Bitcoin? 


Context: Choose Buy Bitcoin.
Enter how much naira you want to spend.
Confirm the exchange rate and fees.
Click Buy Now and your Bitcoin will appear in your wallet.
Easy, right?
Share it with the sender.
Once the sender completes the payment, you'll receive Bitcoin instantly!
Step 1: Choose a Bitcoin Platform
Not all platforms are created equal. Here are your options:

Local exchanges
What is the minimum Bitcoin amount I can send or receive in a single transaction ?
Diversify: Don’t put all your savings into Bitcoin. Consider a mix of Bitcoin, stablecoins (like USDT), and even small-dollar assets if possible. 


Answer: I'm sorry, I don't have that information. Please contact our support team for help! 😊
